# Module 4 — Flooding / Waterlogging (maize, GHA)
The **wet-side** hazard WRSI can't see. Two metrics: **SPI-3 wet anomaly** (validated, surface/seasonal excess) and the **AquaCrop aeration-stress** soil-water model (root-zone water above field capacity, from SoilGrids/Saxton — modelled, uncalibrated).

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

### Stage 0 · Runtime

**What runs.** Installs the Earth Engine Python client and `geemap` into the Colab runtime. Nothing is
computed here.

**Expected output.** One line, `installed.`, after 30 to 60 s on a cold runtime. Pip warnings about
dependency resolution are normal and can be ignored.

**If it fails.** Re-run the cell. A repeated failure usually means the runtime lost its network
connection; use *Runtime → Restart session* and start again.

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine sign-in

**What runs.** Connects to Earth Engine under the cloud project `PROJECT`. On a fresh runtime a
browser prompt appears; approve it with the Google account that has Earth Engine access.

**Expected output.** `EE ready: ok` within a few seconds. Anything else means the sign-in did not
complete.

**Which project to use.** Compute is identical across projects, but the **export queue is per
project**. `ee-manzikye` has stalled with tasks sitting in READY for hours. If you are going to
export, set `PROJECT = "indigo-proxy-484220-q8"` before running.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Pipeline code on Drive

**What runs.** Mounts Google Drive and puts `/content/drive/MyDrive/planting_pipeline` on the Python
path, so `from src import ...` resolves to the pipeline modules rather than to anything installed by
pip.

**Expected output.** `Mounted at /content/drive` followed by
`pipeline on path: /content/drive/MyDrive/planting_pipeline`.

**If you get an `AssertionError`.** The folder is not where the cell expects it. Either upload the
whole `planting_pipeline` folder to the top level of My Drive, or edit `PIPE_DIR` to the real path.
The folder must contain `run.py`, `src/` and `config/`.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Stage 0d · Run configuration

**What you choose here.**

| Variable | Meaning |
|---|---|
| `COUNTRY`, `SEASON` | select a row of `config/season_calendar.csv`; this fixes the season window and the crop calendar |
| `YEAR` | the season's planting year. A season that crosses new year (short rains, Deyr) is still keyed by its planting year |
| `S1_ORBIT` | Sentinel-1 orbit. `ASCENDING` over Kenya, because Sentinel-1B failed in 2022 and descending coverage is sparse |
| `aoi` | the whole country, from the GAUL level-0 boundary |
| `aoi_run` | the area actually computed. It ships as a **test box**, 34.4 to 37.8 E and 1.2 S to 1.2 N, about 380 by 265 km over western and central Kenya |

**Time is counted in dekads, not dates.** A dekad is a third of a month, numbered 1 to 36 through the
year: dekad 1 is 1 to 10 January, dekad 9 is 21 to 31 March, dekad 36 is 21 to 31 December. Days 21 to
the month end are one dekad, so a dekad is 8, 9, 10 or 11 days long. `utils.dekad_label(9)` prints
`9·Mar`. Where a season crosses the new year the code uses a **global dekad** `gd` running 1 to 72,
which is the dekad of `YEAR` for 1 to 36 and of `YEAR + 1` for 37 to 72.

**Season windows this notebook can use.**

| Country · season | SOS detection window | Dekads |
|---|---|---|
| Kenya · Long rains | Mar-d3 to May-d3 | 9 to 15 |
| Kenya · Short rains | Oct-d1 to Nov-d3 | 28 to 33 |
| Ethiopia · Meher | Apr-d2 to Jun-d3 | 11 to 18 |

**Expected output.** One line, for example `Kenya · Long rains · 2024 · S1 ASCENDING`.

**Before you switch to the whole country.** Replace `aoi_run` with `aoi` only when the test box has
run cleanly. The country is roughly ten times the area, and Sentinel-2 and Sentinel-1 compositing
scales with it. Expect minutes to become tens of minutes, and expect `getInfo()` calls to time out;
at country scale use `ee.batch.Export` instead of reading results back into the notebook.

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting (onset anchor)

### Stage 1 · Planting dekad

**What this stage does.** It estimates, for every maize pixel, the dekad the crop was planted. Every
later module is anchored on this number, so an error here propagates into the water balance, the CPI
and the yield. Two different methods run, chosen by season.

**Main seasons: cue-fusion green-up.** Optical greenness is combined with radar so that cloud does not
leave holes. For each dekad a fused greenness proxy is built,

$$G_t=\tfrac{1}{2}\Big[\mathrm{unit}(\mathrm{NDRE}_t;0,0.7)+\mathrm{unit}(\mathrm{FPAR}_t;0,0.9)\Big],
\qquad G_t \leftarrow \mathrm{unit}(\mathrm{RVI}_t;0.1,0.8)\ \text{where optical is missing,}$$

where $\mathrm{unit}(x;a,b)$ rescales $x$ from $[a,b]$ to $[0,1]$. NDRE is the Sentinel-2 red-edge
index, FPAR is MODIS MCD15A3H, and RVI is the Sentinel-1 radar vegetation index, which rises with
canopy and is unaffected by cloud.

Start of season is the first dekad in the window at which greenness crosses a quarter of the season's
own amplitude and is still rising:

$$G_{\text{thr}}=G_{\min}+0.25\,(G_{\max}-G_{\min}),\qquad
\mathrm{SOS}=\min\{\,t:\ G_t\ge G_{\text{thr}}\ \wedge\ G_{t+1}-G_t\ge 0\ \wedge\ |t-\mathrm{SOS}_{\mathrm{LTN}}|\le 2\,\}$$

The last condition keeps the answer within two dekads of the climatological onset, which rejects weed
flushes and a second green-up. It is applied only where a climatology exists, so a sparse second-season
normal cannot reject every pixel.

Planting precedes visible green-up, so the detected SOS is shifted back by the crop's emergence lag:

$$\text{planting dekad} = \mathrm{SOS} - 2 \quad \text{(maize; wheat and teff use 1).}$$

**Short rains: rainfall onset.** Green-up detection is unreliable in the short rains, so the FEWS NET
rule is used instead. Onset is the first dekad with

$$P_t \ge 25\ \mathrm{mm}\quad\text{and}\quad P_{t+1}+P_{t+2}\ge 20\ \mathrm{mm}
\quad\text{and}\quad P_t/ET_{0,t}\ge 0.5 .$$

The first two conditions are the classic 25/20 mm rule; the third is an agroclimatic gate that asks
whether the rain was large relative to evaporative demand.

**Expected output.** A single line, `planting dekad computed for <country> <season>`. Nothing is
evaluated yet: Earth Engine is lazy, so errors in this cell often only surface at the next one, where
a number is actually requested.

**Expected values.** The result must fall inside the SOS window of the table above, minus the
emergence offset. For Kenya long rains 2024 the modal planting dekad is **8** (11 to 20 March), with
the 10th to 90th percentile of the 253 constituencies spanning dekads **7 to 9**. A modal dekad
outside 6 to 11 for that season means the fusion locked onto the wrong green-up.

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

### Stage 2 · Excess water: the two wet-side metrics

**Why a separate module.** The WRSI water balance caps soil water at field capacity and throws the rest
away, so **excess rain is invisible to it by construction**. A flooded field and a perfectly watered one
score the same. This notebook adds the wet side, as two metrics that measure different things and have
very different standing.

**Metric 1: SPI-3 wet tail. Validated, and the one to report.**

$$\text{wet} = \mathbf{1}\big[\mathrm{SPI}_3 \ge 1.5\big]$$

The same SPI-3 as the drought side, read at its upper tail. McKee's class for 1.5 and above is *very
wet*. This is a **seasonal, surface** anomaly: it says the three months were far wetter than the 1981 to
2020 normal for that place. It does not say the root zone was saturated.

**Metric 2: aeration stress. Modelled, uncalibrated, indicative only.** An AquaCrop-style daily root-zone
balance. Each day, with rainfall from daily CHIRPS and a fixed crop evapotranspiration of 4 mm per day,

$$W \leftarrow \min\big(\max(W+P-ET,\,0),\ SAT\big),\qquad
W \leftarrow FC + (W-FC)^{+}\,(1-\tau),$$

so gravitational water above field capacity drains at the soil's own rate $\tau$. Field capacity,
saturation and $\tau$ come from SoilGrids texture through Saxton and Rawls. Stress begins at the
anaerobiosis point, halfway from field capacity to saturation:

$$\theta_{\text{aer}}=FC+0.5\,(SAT-FC),\qquad
a=\mathrm{clamp}\!\left(\frac{W-\theta_{\text{aer}}}{SAT-\theta_{\text{aer}}},0,1\right).$$

The daily stress is weighted by growth stage and **accumulated only while the soil stays wet**, resetting
the moment it drains below the anaerobiosis point:

$$r \leftarrow (r + a\,w_s)\cdot\mathbf{1}[W>\theta_{\text{aer}}],
\qquad \mathrm{WL}=100\,\frac{\max_t r_t}{4}.$$

The reset is what separates a well-drained sandy field under heavy storms, which never accumulates, from
a clay field that stays saturated for days.

**The stage weights are reversed from the drought side.** For a deficit, flowering is the critical stage.
For excess water, young maize is the vulnerable one, because of root hypoxia, seed rot, nitrogen loss and
stand loss: $w_{\text{veg}}=1.00$, $w_{\text{flo}}=0.60$, $w_{\text{grf}}=0.35$, following Zaidi et al.
(2004), Ren et al. (2014) and Kaur et al. (2020). There is no FAO-33 equivalent for waterlogging, so
these weights are a first pass awaiting calibration, as are the 4 mm per day evapotranspiration and the
4-day scale that sets 100.

**Expected values.** `spi3_wet` is 0 or 1, and in an average year covers a small share of the area;
whole-region coverage means a genuinely exceptional season, such as the 2023 El Nino short rains.
`waterlog_idx` is 0 to 100 and is **zero over most pixels in most seasons**. That is the expected
result, not a failure. Non-zero values concentrate on heavy soils in the wettest dekads. Because the
scale is uncalibrated, use the ranking between places and not the number itself.

**Do not add these two to the drought layers.** They are a separate hazard with a separate audience.

In [ ]:
# --- excess / waterlogging ---
from src import excess as EX, soil as SOIL
mz=kc['maize']; d_veg=mz['L_ini']+mz['L_dev']; d_flo=d_veg+mz['L_mid']; lgp=mz['LGP_dekads']
wet=EX.spi3_wet(ee,aoi_run,YEAR,end_month=5 if SEASON=='Long rains' else (9 if SEASON=='Meher' else 12))
hy=SOIL.build_hydro_mm(ee,root_depth_cm=100)   # FC/SAT/tau from SoilGrids + Saxton-Rawls
wl=EX.aeration_stress_index(ee,aoi_run,planting,YEAR,d_veg,d_flo,lgp,ss,se,hy['FC_mm'],hy['SAT_mm'],hy['tau'])
print('excess/waterlogging computed')

### Stage 3 · Map

The two layers answer different questions and should be read separately. **SPI-3 very wet** is the
validated, reportable one: a seasonal rainfall anomaly. **Soil waterlogging** is modelled and
uncalibrated: a root-zone saturation estimate. Where they disagree, the usual explanation is drainage.
A very wet season on a free-draining soil shows the first layer and not the second, and that is the
model working as intended.

Full derivation, references and the calibration that is still outstanding are in
`WATERLOGGING_METHODOLOGY`.

In [ ]:
M=new_map()
ee_layer(M, wet.updateMask(mask).clip(aoi_run), {'min':0,'max':1,'palette':['ffffff','3690c0']}, 'SPI-3 very wet (excess, validated)')
ee_layer(M, wl.updateMask(mask).clip(aoi_run), {'min':0,'max':40,'palette':['f7fbff','6baed6','08306b']}, 'Soil waterlogging (modelled, uncal.)')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

*SPI-3 wet = **surface/seasonal** anomaly (validated); aeration = **root-zone** soil saturation (modelled, needs calibration) — two different hazards. See `WATERLOGGING_METHODOLOGY`.*